# Final Workshop — Orders Medallion Pipeline

**Databricks Fundamentals | Day 1 | ~50 min**

---

## Scenario

You are a data engineer at a retail company. Your manager needs a **fully automated pipeline** that turns raw JSON order files into a daily revenue report — ready for the analytics team by 06:00 every morning.

You will implement the pipeline as three independent notebooks following the **Medallion architecture**, then wire them together into a Databricks Workflow Job.

```
[Volume]  orders_batch.json
      |
      v  Task 01 — lab_task_01_bronze
      |  Read JSON + add metadata -> bronze.lab_orders
      |
      v  Task 02 — lab_task_02_silver
      |  Cast * enrich * filter -> silver.lab_orders
      |
      v  Task 03 — lab_task_03_gold
         Aggregate by date -> gold.lab_daily_orders
```

---

## Layer responsibilities

| Layer | Table | Rule |
|-------|-------|------|
| **Bronze** | `bronze.lab_orders` | Raw data — only metadata columns added |
| **Silver** | `silver.lab_orders` | Typed, enriched, filtered |
| **Gold** | `gold.lab_daily_orders` | Aggregated — one row per day |

---

## Workshop files

| Notebook | Layer | What you build |
|----------|-------|----------------|
| `tasks/lab_task_01_bronze` | Bronze | Raw JSON -> Delta |
| `tasks/lab_task_02_silver` | Silver | Casting, enrichment, filtering |
| `tasks/lab_task_03_gold`   | Gold   | Daily revenue aggregation |

---

## Step 0 — Verify setup


In [ ]:
%run ../../setup/00_setup
print(f"Catalog  : {CATALOG}")
print(f"Bronze   : {BRONZE_SCHEMA}")
print(f"Silver   : {SILVER_SCHEMA}")
print(f"Gold     : {GOLD_SCHEMA}")
print(f"Dataset  : {DATASET_PATH}")

---

## Step 1 — Complete the task notebooks

Open and complete them **in order**. Each notebook:
- starts with `%run ../../setup/00_setup`
- has `dbutils.widgets` — run as-is
- has `# YOUR CODE HERE` cells to fill in
- ends with `dbutils.notebook.exit(json)` returning status

---

## Step 2 — Smoke test

Run this cell after completing all three task notebooks.


In [ ]:
tables = [
    (f"{CATALOG}.{BRONZE_SCHEMA}.lab_orders",     "bronze"),
    (f"{CATALOG}.{SILVER_SCHEMA}.lab_orders",     "silver"),
    (f"{CATALOG}.{GOLD_SCHEMA}.lab_daily_orders", "gold"),
]
all_ok = True
for table, layer in tables:
    try:
        count = spark.table(table).count()
        print(f"  OK  [{layer:6}]  {table}  ->  {count:,} rows")
    except Exception as e:
        print(f"  FAIL [{layer:6}]  {table}  ->  {e}")
        all_ok = False
print()
print("All layers ready" if all_ok else "Some tables missing -- check task notebooks")

---

## Step 3 — Create the Databricks Workflow Job

### 3.1 New Job

1. Left sidebar → **Workflows** → **Create Job**
2. Job name: `lab_orders_pipeline`

---

### 3.2 Task 01 — Bronze Load

| Field | Value |
|-------|-------|
| Task name | `bronze_load` |
| Type | Notebook |
| Path | `notebooks/labs/tasks/lab_task_01_bronze` |
| Cluster | existing all-purpose cluster |
| Parameters | `source_path` -> *(your DATASET_PATH)* |

---

### 3.3 Task 02 — Silver Transform

| Field | Value |
|-------|-------|
| Task name | `silver_transform` |
| **Depends on** | `bronze_load` |
| Type | Notebook |
| Path | `notebooks/labs/tasks/lab_task_02_silver` |

---

### 3.4 Task 03 — Gold Aggregation

| Field | Value |
|-------|-------|
| Task name | `gold_aggregation` |
| **Depends on** | `silver_transform` |
| Type | Notebook |
| Path | `notebooks/labs/tasks/lab_task_03_gold` |

---

### 3.5 Schedule

- **Triggers** tab → **Add trigger** → **Scheduled**
- Cron expression: `0 0 6 * * ?` — every day at 06:00 UTC
- Timezone: `Europe/Warsaw`
- Leave status as **Paused**

---

### 3.6 Run and monitor

1. **Run now** — manual trigger
2. Observe the DAG: `bronze_load -> silver_transform -> gold_aggregation`
3. Click any task → **Output** tab — inspect the `notebook.exit()` JSON
4. If a task fails → **Repair Run** (reruns only failed task + downstream)

---

## Final checklist

- [ ] `lab_task_01_bronze` runs without error
- [ ] `lab_task_02_silver` runs without error
- [ ] `lab_task_03_gold` runs without error
- [ ] Smoke test shows OK for all 3 layers
- [ ] Job: `bronze_load -> silver_transform -> gold_aggregation`
- [ ] Schedule configured (CRON + timezone)
- [ ] Manual run succeeded — all 3 tasks green
- [ ] Checked task output JSON in run logs

> <- [M04: Workflows & Automation](../modules/M04_orchestration_jobs.ipynb) | [M00: Intro](../modules/M00_intro.ipynb) ->
